In [1]:
import keras
from keras import optimizers
from keras.callbacks import ModelCheckpoint
from keras.layers import Convolution2D, MaxPooling2D, ZeroPadding2D
from keras.models import Sequential
from keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image, ImageOps
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import Model, layers, models
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.layers import (Activation, AvgPool2D, BatchNormalization, Concatenate, 
                                     Conv2D, Dense, Dropout, ELU, Flatten, GlobalAveragePooling2D, 
                                     GlobalAvgPool2D, Input, MaxPool2D, MaxPooling2D, ReLU, 
                                     SeparableConv2D, UpSampling2D)
from tensorflow.keras.models import Model, Sequential, model_from_json
from tensorflow.keras.optimizers import RMSprop, SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, MobileNetV2
import tensorflow_datasets as tfds
from tqdm import tqdm

ModuleNotFoundError: No module named 'keras'

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# 1) Load dataset
ds_all, info = tfds.load(
    "plant_village",
    split="train",
    as_supervised=True,   # returns (image, label)
    with_info=True
)

total = info.splits["train"].num_examples
train_size = int(0.8 * total)
val_size = total - train_size

num_classes = info.features["label"].num_classes
print("Number of classes:", num_classes)

# 2) Shuffle then split (reproducible)
ds_all = ds_all.shuffle(
    buffer_size=min(total, 10_000),  # can set total for full shuffle; 10k is faster
    seed=SEED,
    reshuffle_each_iteration=False
)

ds_train_raw = ds_all.take(train_size)
ds_val_raw = ds_all.skip(train_size)

# 3) Augmentation (train only)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# 4) Preprocess for MobileNetV2: resize + float + scale to [-1, 1]
def preprocess_mobilenet(image, label):
    image = tf.image.resize(image, IMG_SIZE)  # Resize image
    image = tf.cast(image, tf.float32)    # Ensure float32
    image = preprocess_input(image)  # MobileNetV2 preprocess: expects values in [0,255] and maps to [-1,1]
    return image, label


def preprocess_mobilenet_train(image, label):
    image, label = preprocess_mobilenet(image, label)
    image = data_augmentation(image, training=True)
    return image, label

# 5) Map → Batch → Prefetch


ds_train = (
    ds_train_raw
      .map(preprocess_mobilenet_train, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(BATCH_SIZE)
      .prefetch(tf.data.AUTOTUNE)
)

ds_val = (
    ds_val_raw
      .map(preprocess_mobilenet, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(BATCH_SIZE)
      .prefetch(tf.data.AUTOTUNE)
)

print(f"Total: {total} | Train: {train_size} | Val: {val_size}")
print("Train element_spec:", ds_train.element_spec)

# Optional sanity check: value range should be ~[-1,1]
x_batch, y_batch = next(iter(ds_train))
print("x batch:", x_batch.shape, x_batch.dtype,
      "min:", tf.reduce_min(x_batch).numpy(),
      "max:", tf.reduce_max(x_batch).numpy())
print("y batch:", y_batch.shape, y_batch.dtype)

In [10]:

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)  # importante: BN en modo inference
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)


model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint("mobilenetv2_v3_head.keras", monitor="val_loss", save_best_only=True),
]

history_head = model.fit(ds_train, validation_data=ds_val, epochs=10, callbacks=callbacks, verbose=2)


Epoch 1/10


KeyboardInterrupt: 

In [4]:
# 1) Descongelar el backbone
base.trainable = True

# 2) Congelar BatchNorm para estabilidad (recomendado en fine-tuning con batch 32 o menor)
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# 3) "Top layers": deja entrenables solo las últimas N capas (ej: 20-40)
N = 30
for layer in base.layers[:-N]:
    layer.trainable = False

# 4) Recompilar con LR bajo (paper usa 1e-4; para fine-tuning suele ir bien 1e-5 a 1e-4)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=2, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint("mobilenetv2_v3_finetuned.keras", monitor="val_loss", save_best_only=True),
]

history_ft = model.fit(ds_train, validation_data=ds_val, epochs=20, callbacks=callbacks_ft, verbose=2)


Epoch 1/20
1358/1358 - 463s - 341ms/step - accuracy: 0.9565 - loss: 0.1272 - val_accuracy: 0.9550 - val_loss: 0.1344 - learning_rate: 1.0000e-05
Epoch 2/20
1358/1358 - 440s - 324ms/step - accuracy: 0.9652 - loss: 0.1010 - val_accuracy: 0.9536 - val_loss: 0.1454 - learning_rate: 1.0000e-05
Epoch 3/20
1358/1358 - 463s - 341ms/step - accuracy: 0.9682 - loss: 0.0901 - val_accuracy: 0.9552 - val_loss: 0.1376 - learning_rate: 1.0000e-05
Epoch 4/20
1358/1358 - 458s - 337ms/step - accuracy: 0.9775 - loss: 0.0672 - val_accuracy: 0.9675 - val_loss: 0.1006 - learning_rate: 1.0000e-06
Epoch 5/20
1358/1358 - 452s - 333ms/step - accuracy: 0.9791 - loss: 0.0634 - val_accuracy: 0.9704 - val_loss: 0.0956 - learning_rate: 1.0000e-06
Epoch 6/20
1358/1358 - 449s - 331ms/step - accuracy: 0.9779 - loss: 0.0649 - val_accuracy: 0.9697 - val_loss: 0.0970 - learning_rate: 1.0000e-06
Epoch 7/20
1358/1358 - 456s - 336ms/step - accuracy: 0.9794 - loss: 0.0606 - val_accuracy: 0.9698 - val_loss: 0.0963 - learning_ra

In [5]:
# Si guardaste en formato Keras (.keras)
best_model = tf.keras.models.load_model("mobilenetv2_v3_finetuned.keras")

# Guardar como SavedModel (útil para convertir a TFLite)
best_model.export("saved_models/mobilenetv2_v3")

INFO:tensorflow:Assets written to: saved_models/mobilenetv2_v3/assets


INFO:tensorflow:Assets written to: saved_models/mobilenetv2_v3/assets


Saved artifact at 'saved_models/mobilenetv2_v3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  5641382672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678700240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678694672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678699280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678700816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678699664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678702352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678702736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678702544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678695056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5678703696: TensorSpec(shape=(), dtype=tf.resource,

In [5]:
# Converter 
converter = tf.lite.TFLiteConverter.from_saved_model("saved_models/mobilenetv2_v3")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model_fp16 = converter.convert()

with open("mobilenetv2_v3_fp16.tflite", "wb") as f:
    f.write(tflite_model_fp16)

W0000 00:00:1772493880.347301 9656306 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772493880.348108 9656306 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-02 20:24:40.352413: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: saved_models/mobilenetv2_v3
2026-03-02 20:24:40.357383: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-02 20:24:40.357391: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: saved_models/mobilenetv2_v3
I0000 00:00:1772493880.403439 9656306 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-03-02 20:24:40.412299: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-02 20:24:40.731449: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: saved_models/mobilenetv2_v3
2026-03-02 20:24:40.807402: I tensorflow/cc/saved

In [6]:
saved_model_dir = "saved_models/mobilenetv2_v3"

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open("mobilenetv2_v3_fp16.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ Saved:", "mobilenetv2_v3_fp16.tflite", "bytes:", len(tflite_model))

✅ Saved: mobilenetv2_v3_fp16.tflite bytes: 4560664


W0000 00:00:1772494099.872749 9656306 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772494099.873894 9656306 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-02 20:28:19.877400: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: saved_models/mobilenetv2_v3
2026-03-02 20:28:19.882811: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-02 20:28:19.882820: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: saved_models/mobilenetv2_v3
2026-03-02 20:28:19.944098: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-02 20:28:20.299912: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: saved_models/mobilenetv2_v3
2026-03-02 20:28:20.380722: I tensorflow/cc/saved_model/loader.cc:466] SavedModel load for tags { serve }; Status: success: OK. Took 503578 microseconds.


In [7]:
converter = tf.lite.TFLiteConverter.from_saved_model("saved_models/mobilenetv2_v3")
tflite_model = converter.convert()

with open("mobilenetv2_v3_fp32.tflite", "wb") as f:
    f.write(tflite_model)

W0000 00:00:1772494164.499545 9656306 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772494164.499577 9656306 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-02 20:29:24.499754: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: saved_models/mobilenetv2_v3
2026-03-02 20:29:24.504335: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-02 20:29:24.504343: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: saved_models/mobilenetv2_v3
2026-03-02 20:29:24.557610: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-02 20:29:24.856905: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: saved_models/mobilenetv2_v3
2026-03-02 20:29:24.936299: I tensorflow/cc/saved_model/loader.cc:466] SavedModel load for tags { serve }; Status: success: OK. Took 436544 microseconds.


In [12]:
ds_all, info = tfds.load(
    "plant_village",
    split="train",
    as_supervised=True,   # returns (image, label)
    with_info=True
)

class_names = info.features["label"].names  # TFDS

with open("labels.txt", "w") as f:
    for name in class_names:
        f.write(name + "\n")